# GristMill — Training Walkthrough

This notebook walks through the full SieveTrainer pipeline interactively:
loading feedback data, extracting features, training, exporting to ONNX, and
running parity validation.

**Prerequisites:** `pip install -e gristmill-ml`

In [ ]:
import logging
import sys
from pathlib import Path

import numpy as np

# Add the package root if running from the notebooks/ directory
sys.path.insert(0, str(Path('..') / 'src'))

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(name)s — %(message)s')
logger = logging.getLogger('walkthrough')

FEEDBACK_DIR = Path('~/.gristmill/feedback').expanduser()
OUTPUT_DIR   = Path('~/.gristmill/models').expanduser()

print('Setup complete.')

In [ ]:
from gristmill_ml.datasets.feedback import FeedbackDataset

train_ds = FeedbackDataset(FEEDBACK_DIR, split='train', min_records=200)
val_ds   = FeedbackDataset(FEEDBACK_DIR, split='val',   min_records=200)

counts = train_ds.class_counts()
print(f'Training set: {len(train_ds)} records')
print(f'Validation set: {len(val_ds)} records')
print('Class distribution (train):')
for label, count in counts.items():
    bar = '#' * (count // 10)
    print(f'  {label:<15} {count:>4}  {bar}')

In [ ]:
from sentence_transformers import SentenceTransformer
from gristmill_ml.training.sieve_trainer import FeatureExtractor, FEATURE_DIM

embedder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
extractor = FeatureExtractor(embedder)

sample_texts = [
    'status',
    'schedule meeting tomorrow at 3pm',
    'summarise the last 10 support tickets',
    'explain why the auth service failed intermittently',
]
sample_sources = ['cli', 'http', 'http', 'http']
sample_priorities = [1, 1, 2, 3]

features = np.stack([
    extractor.extract(t, s, p)
    for t, s, p in zip(sample_texts, sample_sources, sample_priorities)
])

print(f'Feature matrix shape: {features.shape}  (expected [4, {FEATURE_DIM}])')
print('Embedding range (dims 0-383): [{:.3f}, {:.3f}]'.format(features[:, :384].min(), features[:, :384].max()))
print('Metadata features (dims 384-391):')
meta_names = ['log_tc', 'ch_ord', 'priority', 'entity_density', 'q_prob', 'code_prob', 'ttr', 'ambiguity']
for i, name in enumerate(meta_names):
    vals = features[:, 384 + i]
    print(f'  [{384+i}] {name:<16} {vals}')

In [ ]:
from gristmill_ml.training.sieve_trainer import SieveTrainer

trainer = SieveTrainer(
    feedback_dir=FEEDBACK_DIR,
    output_dir=OUTPUT_DIR,
    experiment_name='walkthrough',
    augment=True,
)

# Train for a few epochs to keep this demo fast
result = trainer.train(epochs=3, batch_size=64)

print(f'\nBest val accuracy: {result.best_val_accuracy:.4f} (epoch {result.best_epoch + 1})')

In [ ]:
try:
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(result.train_losses, marker='o', label='Train loss')
    axes[0].set_title('Training Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].legend()
    axes[0].grid(True)

    axes[1].plot(result.val_accuracies, marker='s', color='orange', label='Val accuracy')
    axes[1].set_title('Validation Accuracy')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy')
    axes[1].legend()
    axes[1].grid(True)

    plt.tight_layout()
    plt.show()
    print('Training curves plotted.')
except ImportError:
    print('matplotlib not installed — skipping plot. pip install matplotlib')
    for i, (loss, acc) in enumerate(zip(result.train_losses, result.val_accuracies)):
        print(f'  Epoch {i+1}: loss={loss:.4f}, val_acc={acc:.4f}')

In [ ]:
from gristmill_ml.export.onnx_export import OnnxExporter

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
onnx_path = OUTPUT_DIR / 'intent-classifier-v1-walkthrough.onnx'

exported = OnnxExporter.export_classifier(trainer.model, onnx_path, quantize=True)
print(f'Exported ONNX: {exported}')
print(f'File size: {exported.stat().st_size / 1024:.1f} KB')

In [ ]:
from gristmill_ml.export.validate import validate_classifier_parity
from gristmill_ml.training.sieve_trainer import FEATURE_DIM

dummy = np.random.randn(32, FEATURE_DIM).astype(np.float32)
val_result = validate_classifier_parity(trainer.model, exported, dummy)

passed = val_result['passed'] if isinstance(val_result, dict) else val_result.passed
max_diff = val_result['max_diff'] if isinstance(val_result, dict) else val_result.max_diff
agreement = val_result.get('label_agreement_rate', 1.0) if isinstance(val_result, dict) else val_result.label_agreement_rate

status = 'PASS' if passed else 'FAIL'
print(f'Parity check: [{status}]')
print(f'  max_diff       = {max_diff:.8f}')
print(f'  label_agreement = {agreement:.4f}')

In [ ]:
import onnxruntime as ort

sess = ort.InferenceSession(str(exported), providers=['CPUExecutionProvider'])

LABEL_NAMES = ['LOCAL_ML', 'RULES', 'HYBRID', 'LLM_NEEDED']

print('Sample inference results:')
print(f'{"Text":<45} {"Routing decision"}')
print('-' * 65)

for text, source, priority in zip(sample_texts, sample_sources, sample_priorities):
    fvec = extractor.extract(text, source, priority)[np.newaxis].astype(np.float32)
    logits = sess.run(None, {'features': fvec})[0]
    pred_idx = int(logits.argmax())
    conf = float(__import__('torch').softmax(__import__('torch').from_numpy(logits), dim=1).numpy()[0, pred_idx])
    label = LABEL_NAMES[pred_idx]
    short_text = text[:43]
    print(f'{short_text:<45} {label} ({conf:.2%})')